# 과제 1.1  Pauli 연산자

## 목표: Pauli 연산자 정의하기

**개요:** 이 노트북에서는 Pauli 연산자와 그 초기화 방법, 속성 및 메서드를 다룹니다.

Pauli 연산자는 단일 큐비트 및 다중 큐비트 연산을 표현하는 양자 컴퓨팅의 기본 구성 요소입니다. Qiskit의 `Pauli` 클래스는 이러한 연산자를 생성하고 조작하는 다양한 방법을 제공합니다.

In [1]:
# 설정: 필요한 라이브러리 불러오기
from qiskit.quantum_info import Pauli, ScalarOp
import numpy as np
from qiskit import QuantumCircuit

print("라이브러리를 성공적으로 불러왔습니다.")

라이브러리를 성공적으로 불러왔습니다.


### 초기화 방법

#### 1. Pauli 문자열 표현

Pauli 행렬을 나타내는 문자열로 Pauli 연산자를 생성할 수 있습니다:
- `I`: 항등 연산자
- `X`: Pauli-X(비트 반전) 연산자
- `Y`: Pauli-Y 연산자
- `Z`: Pauli-Z(위상 반전) 연산자

참고: Qiskit은 <b>Little Endian 표기법</b>을 사용하므로 문자열 표현에서 큐비트 (n-1)은 가장 왼쪽에, 큐비트 0은 가장 오른쪽에 놓입니다.

위상은 ` `(기본값), `-`, `i`, `-i` 중 하나로 지정할 수 있습니다.

In [33]:
# 문자열 표현으로 Pauli 연산자 생성
p = Pauli('XZ')      # 위상 없음(기본값 +1)
p2 = Pauli('iYY')    # 허수 위상 i 포함
p3 = Pauli('-iZX')   # 음의 허수 위상 -i 포함

print(f"P:{p}, P2:{p2}, P3:{p3}")

P:XZ, P2:iYY, P3:-iZX


#### 2. Boolean 배열 표현

NumPy Boolean 배열을 사용해서도 Pauli 연산자를 생성할 수 있습니다:
- `z` 배열: Z 성분을 지정합니다(True = Z 연산자, False = I 연산자).
- `x` 배열: X 성분을 지정합니다(True = X 연산자, False = I 연산자).

각 큐비트 위치의 실제 Pauli 연산자는 다음과 같이 결정됩니다:
- (z=False, x=False) → I(항등 연산자)
- (z=False, x=True) → X
- (z=True, x=False) → Z
- (z=True, x=True) → Y(Y = iXZ이므로)

In [55]:
# Boolean 배열을 사용하여 Z 성분과 X 성분 정의
# 큐비트 0에는 I, 큐비트 1에는 X, 큐비트 2에는 Z, 큐비트 3에는 Y 적용
z = np.array([False,False,True,True])  
x = np.array([False,True,False,True])  

pauli_array_rep = Pauli((z, x))

print(f"Boolean 배열로 생성한 Pauli 연산자: {pauli_array_rep}")
# 배열의 큐비트 순서는 문자열 표현과 비교하면 반대임에 유의

Boolean 배열로 생성한 Pauli 연산자: YZXI


배열에서는 왼쪽부터 인덱싱하지만 Qiskit에서는 오른쪽부터 큐비트 0을 시작하므로, 출력 순서가 반대라는 점에 유의하세요.

#### 3. 양자 회로 표현

Pauli 게이트(I, X, Y, Z)만 포함된 양자 회로에서 Pauli 연산자를 추출할 수 있습니다. 회로를 분석하여 이에 대응하는 Pauli 연산자를 구성합니다.

In [ ]:
# Pauli 게이트로 양자 회로 생성
qc = QuantumCircuit(4)
qc.id(0) # 큐비트 0에 I 게이트 적용(필수는 아니며 설명을 위해 사용)
qc.z(1)  # 큐비트 1에 Z 게이트 적용
qc.x(2)  # 큐비트 2에 X 게이트 적용
qc.y(3)  # 큐비트 3에 Y 게이트 적용
# 다른 게이트를 사용하면 안 됩니다. 아래 줄의 주석을 해제하면 오류가 발생합니다.
# qc.cx(0,1)


# 회로에서 Pauli 연산자 추출
pauli_quantum_circuit = Pauli(qc)

print(f"양자 회로에서 생성한 Pauli 연산자: {pauli_quantum_circuit}")

양자 회로에서 생성한 Pauli 연산자: YXZI


#### 4. `ScalarOp`에서 생성

Pauli 연산자는 스칼라 연산과 결합할 수 있습니다. `ScalarOp`는 항등 연산자의 스칼라배를 나타내며, Pauli 연산자와 합성하여 전역 위상을 적용할 수 있습니다.

In [ ]:
# 스칼라 연산자(스케일된 항등 연산자) 생성
# Pauli 연산자에는 1, -1j, -1, 1j만 곱할 수 있음
scalar_op = ScalarOp(dims=(2,2,2,2), coeff=-1j)  # -1*j

# Pauli 연산자 생성
pauli_scalarop = Pauli('YXZI')

# 스칼라 연산자와 Pauli 연산자 합성
# Pauli 연산자에 스칼라 계수를 적용함
p = scalar_op.compose(pauli_scalarop)

print(f"스칼라 적용 전 Pauli 연산자: {pauli_scalarop}")

print(f"ScalarOp에서 생성한 Pauli 연산자: {p}")

스칼라 적용 전 Pauli 연산자: YXZI
ScalarOp에서 생성한 Pauli 연산자: -iYXZI


### 표현 방법

#### 1. 행렬 표현

`to_matrix()` 메서드는 Pauli 연산자를 행렬 표현으로 변환합니다. 다중 큐비트 Pauli 연산자의 경우 개별 Pauli 행렬의 텐서곱을 반환합니다.

In [ ]:
# Pauli 연산자를 행렬 표현으로 변환
p = Pauli('ZX')
p_matrix = p.to_matrix()

print(f"행렬 형식:\n{p_matrix}")

행렬 형식: 
 [[ 0.+0.j  1.+0.j  0.+0.j  0.+0.j]
 [ 1.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j -1.+0.j]
 [ 0.+0.j  0.+0.j -1.+0.j  0.+0.j]]


#### 2. 문자열 표현

Pauli 연산자는 읽기 쉬운 문자열 레이블로 변환할 수 있습니다. `to_label()` 메서드는 표준 문자열 표현을 제공합니다.

In [26]:
# 문자열 표현
print(f"문자열 형식:{str(p)}")

print(f"레이블 형식:{p.to_label()}")

문자열 형식:ZX
레이블 형식:ZX


### 속성

`Pauli` 클래스는 연산자의 속성에 접근할 수 있는 여러 속성을 제공합니다. 그중 일부는 다음과 같습니다:
- `dim`: 연산자의 차원
- `num_qubits`: 큐비트 수
- `num_clbits`: 고전 비트 수(Pauli 연산자에서는 항상 0)
- `phase`: 연산자의 위상(0, 1, 2, 3은 각각 +1, i, -1, -i를 나타냄)
- `x`: Boolean 배열로 표현한 X 성분
- `z`: Boolean 배열로 표현한 Z 성분

In [32]:
# Pauli 연산자의 속성에 접근
print(f"차원: {p.dim}")  # 연산자 행렬의 차원
print(f"고전 비트: {p.num_clbits}, 큐비트: {p.num_qubits}")  # 큐비트 수
print(f"X: {p.x}, Z: {p.z}, 위상: {p.phase}")  # 성분과 위상

차원: (4, 4)
고전 비트: 0, 큐비트: 2
X: [ True False], Z: [False  True], 위상: 0


### 메서드

`Pauli` 클래스는 연산자를 조작하고 분석하는 다양한 메서드를 제공합니다. 그중 일부는 다음과 같습니다.

#### `adjoint`(수반)

Pauli 연산자의 수반(켤레 전치)을 반환합니다. Pauli 연산자는 에르미트 연산자이므로 수반이 자기 자신과 같지만, 위상은 달라질 수 있습니다.

In [ ]:
# Pauli 연산자의 수반(켤레 전치) 구하기
p.adjoint()
print(f"Pauli 연산자: {p3}, 수반: {p3.adjoint()}")

Pauli 연산자:-iZX, 수반: iZX


#### `anticommutes`(반교환 여부)

두 Pauli 연산자가 반교환하는지 확인합니다. 두 연산자 A와 B가 AB = -BA를 만족하면 서로 반교환합니다.

In [37]:
# 두 Pauli 연산자가 반교환하는지 확인
p = Pauli('X')      # X
p2 = Pauli('Y')     # Y
print(f"{p}와 {p2}의 반교환 여부: {p.anticommutes(p2)}")

X와 Y의 반교환 여부: True


#### `compose`(합성)

두 Pauli 연산자의 합성(행렬곱)을 반환합니다. Pauli 연산자를 합성하면 위상 인자를 제외하고 또 다른 Pauli 연산자가 됩니다.

In [40]:
# 두 Pauli 연산자 합성(행렬곱)

print(f"{p}와 {p2}를 합성한 결과: {p.compose(p2)}")
# 행렬 연산자에서 합성(&)은 기본적으로 왼쪽 행렬곱으로 정의됨. 즉, A & B == A.compose(B)

X와 Y를 합성한 결과: -iZ


#### `conjugate`(복소켤레)

Pauli 연산자의 복소켤레를 반환합니다. 허수 위상이 없는 실수 Pauli 연산자의 경우 원래 연산자와 같습니다.

In [41]:
# Pauli 연산자의 복소켤레 구하기
p = Pauli('ZX')
print(f"{p}의 복소켤레: {p.conjugate()}")
print(f"{p2}의 복소켤레: {p2.conjugate()}")

ZX의 복소켤레: ZX
Y의 복소켤레: -Y


#### `delete`(삭제)

지정한 큐비트를 삭제한 Pauli 연산자를 반환합니다. 이에 따라 연산자의 큐비트 수가 줄어듭니다.

In [44]:
# Pauli 연산자의 지정한 인덱스에서 큐비트 삭제
qubit_index = 0
delete_0 = p.delete(qubit_index)

print(f"{p}에서 큐비트 {qubit_index}을(를) 삭제한 결과: {delete_0}")

ZX에서 큐비트 0을 삭제한 결과: Z


#### `insert`(삽입)

지정한 위치에 큐비트를 추가로 삽입한 Pauli 연산자를 반환합니다.

In [47]:
# 한 Pauli 연산자의 특정 큐비트 위치에 다른 Pauli 연산자 삽입
qubit_index = 1
insert_0 = delete_0.insert(qubit_index, Pauli('X'))

print(f"{delete_0}의 큐비트 {qubit_index} 위치에 X를 삽입한 결과: {insert_0}")

Z의 큐비트 1 위치에 X를 삽입한 결과: XZ


#### `to_instruction`(명령어로 변환)

Pauli 연산자를 양자 회로에서 사용할 수 있는 양자 명령어로 변환합니다.

In [48]:
# Pauli 연산자를 양자 명령어로 변환
p.to_instruction()

Instruction(name='pauli', num_qubits=2, num_clbits=0, params=['ZX'])

---
## 요약
---

이 노트북에서는 다음 내용을 다루었습니다:

## Pauli 연산자:
1. Pauli 문자열, Boolean 배열, 양자 회로 또는 `ScalarOp`를 사용하여 **초기화할 수 있습니다**.
2. `to_matrix()`를 사용한 행렬 또는 `to_label()`을 사용한 문자열로 **표현할 수 있습니다**.
3. 속성에 접근하거나 이를 변경하는 데 사용할 수 있는 다양한 **속성과 메서드를 제공합니다**.



---

## Practice Questions (연습문제)

**1) What is the output of the following Qiskit code?**

*(다음 Qiskit 코드의 출력은 무엇인가요?)*

```
from qiskit.quantum_info import Pauli
import numpy as np

z = np.array([True, False, True])   
x = np.array([False, True, True])
p = Pauli((z, x))
print(p)
```

A) ZXY

B) YXZ

C) IY

D) XZY

E) -iYZ


***Answer:*** *(정답)*
<Details>
<br/>
B) YXZ

Qubit 0: (Z=True, X=False) → Z operator
(큐비트 0: (Z=True, X=False) → Z 연산자)

Qubit 1: (Z=False, X=True) → X operator
(큐비트 1: (Z=False, X=True) → X 연산자)

Qubit 2: (Z=True, X=True) → Y operator
(큐비트 2: (Z=True, X=True) → Y 연산자)
</Details>

---

**2. Which initialization method will create a Pauli operator equivalent to the single-qubit Y gate (up to global phase)?**

*(전역 위상을 제외하고 단일 큐비트 Y 게이트와 동등한 Pauli 연산자를 생성하는 초기화 방법은 무엇인가요?)*

A) ```Pauli('Y')```

B) ```Pauli(([True], [True]))```

C) ```Pauli(([False], [True]))```

D)  ```qc = QuantumCircuit(2)```
       
       qc.y(1)
       
       pauli_quantum_circuit = Pauli(qc)


E) Both A and B (A와 B 모두)

***Answer:*** *(정답)*
<Details>
<br/>
E) Both A and B — `Pauli('Y')` and `Pauli(([True], [True]))` both create the Y operator, while C creates X and D creates YI.

(E) A와 B 모두 — `Pauli('Y')`와 `Pauli(([True], [True]))`는 모두 Y 연산자를 생성합니다. 반면 C는 X를, D는 YI를 생성합니다.)
</Details>

---

**3) What is the output of the code snippet below?**

*(아래 코드 조각의 출력은 무엇인가요?)*

    from qiskit.quantum_info import Pauli
    p = Pauli('ZYX')
    result = p.delete([0,2])
    print(result)

A) XYZ

B) XZ

C) Y

D) ZYX

E) ZX

***Answer:*** *(정답)*
<Details>
<br/>
C) Y — qubits 0 and 2 are deleted; qubit 1 remains, which is Y.

(C) Y — 큐비트 0과 2가 삭제되고, Y에 해당하는 큐비트 1만 남습니다.)
</Details>